# EDA — IEEE-CIS, the numbers behind the design decisions

This notebook is **not the deliverable**. Everything here that mattered was
moved into `src/` behind a test. What it is for: showing where the constants
in `triage/evidence.py`, the V-column subset in `features/spec.py`, and the
embargo length in `features/offline.py` actually came from.

Run order: `make features` first — this reads the processed splits, not the
raw CSVs.

In [ ]:
import json

import numpy as np
import pandas as pd

from fraudsight.config import settings
from fraudsight.features.spec import AMOUNT, LABEL, TIME

train = pd.read_parquet(settings.data_dir / "processed" / "train.parquet")
summary = json.loads((settings.artifact_dir / "feature_summary.json").read_text())
print(summary["bounds_human"])
print(f"{len(train):,} training rows, {train[LABEL].mean():.4%} fraud")

## 1. Class imbalance — why AUC-PR, not ROC-AUC

At a ~3.4% positive rate the negative class dominates ROC-AUC, which reads
around 0.88 for a model that is mediocre where it matters. AUC-PR moves with
the quality that affects the decision.

In [ ]:
print(train[LABEL].value_counts())
print(f"\nnegatives per positive: {(1 - train[LABEL].mean()) / train[LABEL].mean():.1f}")

## 2. Fraud rate over time — why the split is chronological

The rate is not stationary. A random split lets the model see the future and
reports a number the production system cannot reproduce.

In [ ]:
day = (train[TIME] // 86_400).astype(int)
weekly = train.assign(week=day // 7).groupby("week")[LABEL].agg(["mean", "count"])
print(weekly.head(16))
print(f"\nweekly fraud rate: min {weekly['mean'].min():.4f} max {weekly['mean'].max():.4f}")
print(f"ratio max/min: {weekly['mean'].max() / weekly['mean'].min():.2f}x")

## 3. Amount — where `AMOUNT_SPIKE_RATIO = 2.0` comes from

The rule fires when a transaction exceeds 2× the card's own 95th percentile.
The threshold is a round number chosen from this distribution, not a tuned
constant — tuning it against the eval set would be fitting the evidence layer
to the answers.

In [ ]:
amt = train[AMOUNT].astype(float)
print(amt.describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]))
print(f"\nmean amount | fraud:     ${amt[train[LABEL] == 1].mean():,.2f}")
print(f"mean amount | legitimate: ${amt[train[LABEL] == 0].mean():,.2f}")

cents = ((amt - np.floor(amt)) * 100).round()
round_amt = cents == 0
print(f"\nfraud rate on round amounts:     {train.loc[round_amt, LABEL].mean():.4%}")
print(f"fraud rate on non-round amounts: {train.loc[~round_amt, LABEL].mean():.4%}")

## 4. Overnight transactions — where `NIGHT_HOURS = (0, 6)` comes from

In [ ]:
by_hour = train.groupby(train["hour_of_day"].astype(int))[LABEL].agg(["mean", "count"])
print(by_hour)
night = train["hour_of_day"] < 6
print(f"\nfraud rate 00:00-05:59: {train.loc[night, LABEL].mean():.4%}")
print(f"fraud rate 06:00-23:59: {train.loc[~night, LABEL].mean():.4%}")
print(f"lift: {train.loc[night, LABEL].mean() / train.loc[~night, LABEL].mean():.2f}x")

## 5. Missingness is signal, not noise

The identity table joins to only a fraction of transactions, and whether it
joins is itself predictive. This is why `has_identity_record` and
`missing_field_count` are features rather than something to impute away.

In [ ]:
has_id = train["has_identity_record"] > 0
print(f"transactions with an identity record: {has_id.mean():.2%}")
print(f"  fraud rate with:    {train.loc[has_id, LABEL].mean():.4%}")
print(f"  fraud rate without: {train.loc[~has_id, LABEL].mean():.4%}")

q = train["missing_field_count"].quantile([0.1, 0.5, 0.9])
print(f"\nmissing_field_count quantiles:\n{q}")

## 6. The V-block — why 94 columns and not 339

The V-columns are heavily collinear. The subset in `features/spec.py` was
chosen to span the correlation clusters rather than to take the first N. The
measured cost of the full block is ~4× memory and training time for ~0.003
AUC-PR, which is inside the run-to-run spread — recorded in `docs/scope.md`.

In [ ]:
v_cols = [c for c in train.columns if c.startswith("V")]
corr = train[v_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack().sort_values(ascending=False)
print(f"{len(v_cols)} V-columns retained")
print(f"pairs with |r| > 0.95: {(pairs > 0.95).sum()}")
print(f"\nmost collinear surviving pairs:\n{pairs.head(8)}")

## 7. Entity concentration — why velocity is evidence, not a feature

A handful of `card1` values carry a large share of volume. Those are the cards
where velocity counters say something. They are still not model features: the
scorer trained on a flat table with no session context, so feeding it velocity
at serve time would be train/serve skew. See `docs/scope.md`.

In [ ]:
per_card = train.groupby("card1").size().sort_values(ascending=False)
print(f"distinct cards: {len(per_card):,}")
print(f"median transactions per card: {per_card.median():.0f}")
print(f"p99: {per_card.quantile(0.99):.0f}   max: {per_card.max():,}")
print(f"top 1% of cards carry {per_card.head(len(per_card) // 100).sum() / len(train):.1%} of volume")

## 8. Fairness slices

Reported because hiding a disparity is worse than having one. Slices by
transaction size, product code and card type — the segments this dataset
actually supports. It carries no protected attributes, which is itself a
limitation worth stating: a real deployment must slice by the attributes that
matter and this dataset cannot.

In [ ]:
from fraudsight.scoring.dataset import artifact, load_split, load_stats
from fraudsight.scoring.serve import Scorer

scorer = Scorer.load()
test = load_split("test", load_stats())
probs = scorer.score_frame(test.X)

def slice_metrics(mask: np.ndarray, name: str) -> dict:
    y, p = test.y[mask], probs[mask]
    if y.size < 200:
        return {"slice": name, "n": int(y.size), "note": "too few rows to report"}
    approved = p < scorer.lo
    blocked = p >= scorer.hi
    fraud = y == 1
    return {
        "slice": name,
        "n": int(y.size),
        "fraud_rate": round(float(fraud.mean()), 4),
        "recall": round(float(1 - (approved & fraud).sum() / max(fraud.sum(), 1)), 4),
        "false_block_rate": round(float((blocked & ~fraud).sum() / max((~fraud).sum(), 1)), 5),
        "review_rate": round(float((~approved & ~blocked).mean()), 4),
    }

rows = []
amt_test = test.X[AMOUNT].astype(float).to_numpy()
for lo, hi, label in [(0, 50, "<$50"), (50, 250, "$50-250"), (250, 1e9, ">$250")]:
    rows.append(slice_metrics((amt_test >= lo) & (amt_test < hi), f"amount {label}"))
for product in test.X["ProductCD"].astype(str).unique()[:5]:
    rows.append(slice_metrics((test.X["ProductCD"].astype(str) == product).to_numpy(), f"product {product}"))
for card_type in test.X["card6"].astype(str).unique()[:4]:
    rows.append(slice_metrics((test.X["card6"].astype(str) == card_type).to_numpy(), f"card6 {card_type}"))

fairness = pd.DataFrame(rows)
print(fairness.to_string(index=False))

reported = fairness.dropna(subset=["recall"]) if "recall" in fairness else fairness
if "recall" in reported and len(reported) > 1:
    spread = reported["recall"].max() - reported["recall"].min()
    print(f"\nrecall spread across reported slices: {spread:.3f}")
    print("Any spread above ~0.10 is a disparity worth naming in the README.")

In [ ]:
fairness.to_json(settings.artifact_dir / "fairness_slices.json", orient="records", indent=2)
print(f"written -> {settings.artifact_dir / 'fairness_slices.json'}")